Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 ## tool_executor
 **Runs the tools the agent selected and returns results as `ToolMessage` objects back into state.**
 ---

In [ ]:
import time
from langchain_core.messages import ToolMessage
from app.agent.state import AgentState
from app.agent.tools.web_search import web_search
from app.agent.tools.calculator import calculator
from app.agent.tools.email_sender import send_email
from app.core.logging import get_logger

log = get_logger(__name__)

## as we implemeted in the `tools` folder you can go and take a look on this
 ## Sequential, not parallel
 Tools run in a `for` loop — one at a time. the actual implementation is sequential. tool calls are rarely independent enough to benefit from parallelism, and sequential execution is easier to debug.
 ## Dict vs object handling
 LangChain tool calls come back as either dicts or objects depending on the model version:
 ```python
 if isinstance(tool_call, dict):
     tool_name = tool_call.get("name")
 else:
     tool_name = getattr(tool_call, "name", None)
 ```
 Both paths normalize to the same local variables before execution. Clean.
 ## Error handling per tool
 Each tool call is wrapped in its own `try/except`. If `web_search` fails, `calculator` still runs. The error goes back as a `ToolMessage` with the error string as content — the agent sees it on the next loop iteration and can decide how to respond. One bad tool doesn't abort the whole executor.
 Unknown tool names also produce a `ToolMessage` error rather than raising — the `tool_call_id` must be returned for every tool call the LLM made, otherwise LangChain raises a validation error about unmatched tool calls.
 ---
 ### Return shape
 ```python
 "messages": messages + tool_messages,
 ```
 Returns the full message list plus the new `ToolMessage` objects appended. Combined with the `add_messages` reducer in `state.py`, this means the agent's next invocation sees the complete conversation including all tool results.
 `tool_call_history` accumulates across iterations — the planner reads the last 2 entries to avoid suggesting tools that already failed.

In [ ]:
ALL_TOOLS = {
    "web_search": web_search,
    "calculator": calculator,
    "send_email": send_email,
}

async def tool_executor_node(state: AgentState) -> dict:
    """
    Run tools suggested by the last AIMessage.
    Returns updated messages (original + tool results) and trace information.
    """
    messages = list(state.get("messages", []))
    if not messages:
        return {
            "messages": messages,
            "workflow_trace": state.get("workflow_trace", []),
            "tool_call_history": state.get("tool_call_history", []),
        }

    last_message = messages[-1]
    tool_calls = getattr(last_message, "tool_calls", []) or []

    if not tool_calls:
        return {
            "messages": messages,
            "workflow_trace": state.get("workflow_trace", []),
            "tool_call_history": state.get("tool_call_history", []),
        }

    tool_messages = []
    executed_results = []

    for tool_call in tool_calls:
        # Extract tool call details (supports both dict and object)
        if isinstance(tool_call, dict):
            tool_name = tool_call.get("name")
            tool_args = tool_call.get("args") or {}
            tool_call_id = tool_call.get("id")
        else:
            tool_name = getattr(tool_call, "name", None)
            tool_args = getattr(tool_call, "args", None) or {}
            tool_call_id = getattr(tool_call, "id", None)

        tool = ALL_TOOLS.get(tool_name)

        if tool is None:
            log.error("tool_not_found: %s", tool_name)
            error_message = f"Tool not available: {tool_name}"
            tool_messages.append(
                ToolMessage(content=error_message, tool_call_id=tool_call_id, name=tool_name)
            )
            executed_results.append({
                "tool": tool_name,
                "error": error_message,
            })
            continue


        try:
            log.info("executing_tool : %s args: %s", tool_name, tool_args)
            result = await tool.ainvoke(tool_args)

        
            tool_messages.append(
                ToolMessage(content=str(result), tool_call_id=tool_call_id, name=tool_name)
            )
            executed_results.append({
                "tool": tool_name,
                "args": tool_args,
                "result": result,
            })

        except Exception as e:
            log.exception("tool_execution_error: %s", tool_name)
            error_message = f"Error during {tool_name} tool execution: {e}"
            tool_messages.append(
                ToolMessage(content=error_message, tool_call_id=tool_call_id, name=tool_name)
            )
            executed_results.append({
                "tool": tool_name,
                "args": tool_args,
                "error": str(e),
            })

    # Update workflow trace
    workflow_trace = list(state.get("workflow_trace", []))
    workflow_trace.append({
        "node": "tool_executor",
        "status": "completed",
        "executed_results": executed_results,
        "timestamp": time.time(),
    })

    # Update tool call history
    tool_call_history = list(state.get("tool_call_history", []))
    tool_call_history.append(executed_results)

    return {
        "messages": messages + tool_messages,
        "workflow_trace": workflow_trace,
        "tool_call_history": tool_call_history,
    }